In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arpitagupta11/superstore-csv/Sample - Superstore.csv


In [2]:
# Load dataset
df = pd.read_csv('/kaggle/input/datasets/arpitagupta11/superstore-csv/Sample - Superstore.csv',
                encoding='windows-1252')

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
# Normalize column names; convert names to lowercase, replace spacs with underscores,
# strip whitespace

df.columns=(
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-','_')
)
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [4]:
# Parse Dates; string date columns into datetime64 and 
# calculate shipping duration

df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

# Optional: Derive delivery duration in days
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

# Sanity check: Ensure ship date is never before order date
invalid_dates = df[df['shipping_days'] < 0]
print(f"Invalid date rows: {len(invalid_dates)}")

Invalid date rows: 0


In [5]:
# Clean postal_code: fill missing, convert from float to 5 digit padded string
df['postal_code']=(
    df['postal_code']
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(5)
)


In [6]:
# check nulls per column
print(df.isnull().sum())

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
shipping_days    0
dtype: int64


In [7]:
# Clean up accidental columns and fix postal_code
df['postal_code'] = df['postal_code'].fillna(0).astype(int).astype(str).str.zfill(5)
df = df.drop(columns=[col for col in ['postal_cade', 'ship-date'] if col in df.columns])

In [8]:
# check nulls per column
print(df.isnull().sum())

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
shipping_days    0
dtype: int64


In [9]:
# Check for and handle duplicates
# Check exact row duplicates
exact_dupes = df.duplicated().sum()
print(f"Exact row duplicates: {exact_dupes}")

# Check line-item duplicates (same order containing the exact same product ID multiple times)
line_item_dupes = df.duplicated(subset=['order_id', 'product_id'], keep=False).sum()
print(f"Duplicate product line-items within orders: {line_item_dupes}")


Exact row duplicates: 0
Duplicate product line-items within orders: 16


In [10]:
# Inspect duplicates
# view all 16 rows side by side, sorted by order and product
duplicate_mask = df.duplicated(subset=['order_id', 'product_id'], keep=False)
df[duplicate_mask].sort_values(by=['order_id','product_id'])[['order_id','product_id', 'product_name', 'sales', 'quantity', 'discount', 'profit']]

,order_id,product_id,product_name,sales,quantity,discount,profit
6498,CA-2015-103135,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",135.090,9,0.0,62.1414
6500,CA-2015-103135,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",90.060,6,0.0,41.4276
350,CA-2016-129714,OFF-PA-10001970,Xerox 1881,24.560,2,0.0,11.5432
352,CA-2016-129714,OFF-PA-10001970,Xerox 1881,49.120,4,0.0,23.0864
1300,CA-2016-137043,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",572.760,6,0.0,166.1004
1301,CA-2016-137043,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",286.380,3,0.0,83.0502
9168,CA-2016-140571,OFF-PA-10001954,Xerox 1964,319.760,14,0.0,147.0896
9169,CA-2016-140571,OFF-PA-10001954,Xerox 1964,45.680,2,0.0,21.0128
7881,CA-2017-118017,TEC-AC-10002006,Memorex Micro Travel Drive 16 GB,76.752,6,0.2,10.5534
7882,CA-2017-118017,TEC-AC-10002006,Memorex Micro Travel Drive 16 GB,102.336,8,0.2,14.0712


In [11]:
# Handle exact duplicate rows 3405 & 3406
# 1. Identify columns excluding row_id to find identical business entries
cols_to_compare = [col for col in df.columns if col != 'row_id']

# 2. Drop the single clone entry
df = df.drop_duplicates(subset=cols_to_compare).reset_index(drop=True)

# Verify: Duplicate line items should now drop from 16 to 14 (7 pairs)
remaining_dupes = df.duplicated(subset=['order_id', 'product_id'], keep=False).sum()
print(f"Remaining line-item duplicates to aggregate: {remaining_dupes}")

Remaining line-item duplicates to aggregate: 14


In [12]:
# Aggregate the remaining 
# Grouping keys and aggregations
group_cols = ['order_id', 'product_id']
sum_cols = ['sales', 'quantity', 'profit']

metadata_cols = [
    col for col in df.columns 
    if col not in group_cols + sum_cols + ['row_id', 'discount']
]

agg_rules = {col: 'first' for col in metadata_cols}
agg_rules.update({col: 'sum' for col in sum_cols})
agg_rules['discount'] = 'mean'

# Aggregate
df = df.groupby(group_cols, as_index=False).agg(agg_rules)

# Verify final line-item duplicates are 0
print(f"Final line-item duplicates: {df.duplicated(subset=group_cols).sum()}")

Final line-item duplicates: 0


In [13]:
# Check logical consistency
# Verify State to Region mapping consistency
geo_check = df.groupby('state')['region'].nunique()
inconsistent_states = geo_check[geo_check > 1]
print(f"States mapped to multiple regions: {len(inconsistent_states)}")

# Check for impossible numeric values (negative sales or quantity)
invalid_numerics = df[(df['sales'] < 0) | (df['quantity'] <= 0) | (df['discount'] < 0) | (df['discount'] > 1)]
print(f"Rows with invalid sales/quantity/discount values: {len(invalid_numerics)}")

df.info()

States mapped to multiple regions: 0
Rows with invalid sales/quantity/discount values: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9986 entries, 0 to 9985
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       9986 non-null   object        
 1   product_id     9986 non-null   object        
 2   order_date     9986 non-null   datetime64[ns]
 3   ship_date      9986 non-null   datetime64[ns]
 4   ship_mode      9986 non-null   object        
 5   customer_id    9986 non-null   object        
 6   customer_name  9986 non-null   object        
 7   segment        9986 non-null   object        
 8   country        9986 non-null   object        
 9   city           9986 non-null   object        
 10  state          9986 non-null   object        
 11  postal_code    9986 non-null   object        
 12  region         9986 non-null   object        
 13  category       9986 non-null   obj